# Musicm8 v10 — learned whole-track composer

The music is now composed by the **MIDI-LLM learned text-to-MIDI model**, not the old rule-based note generator. Musicm8 asks it for a complete multitrack arrangement, selects the stronger candidate, locks the entire performance to one shared genre groove, and renders the MIDI through sampled SoundFont instruments. Lead vocals use SoulX phrase chunks with explicit words/phonemes and a vocal score derived from the harmony and lead that the AI actually composed.

The first v10 music run downloads about 3.5 GB of MIDI-LLM weights to `MyDrive/Musicm8/work/midi_llm_models/`; later Colab sessions reuse them.

In [ ]:
# ============================================================
# MUSICM8 V10 — ONE CLICK COMPLETE SONG
# ============================================================
import os, sys, json, shutil, subprocess, secrets
from pathlib import Path

IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
FIXED_SEED = None
SEED = int(FIXED_SEED) if FIXED_SEED is not None else secrets.randbelow(2_000_000_000)
print(f"🎲 MUSICM8 SONG SEED: {SEED}")

# MIDI-LLM composes this many complete candidates and Musicm8 selects the strongest.
COMPOSER_CANDIDATES = 2

# Paste your exact words here. Leave blank for AI-written lyrics.
CUSTOM_LYRICS = r"""
""".strip()

# Optional authorized timbre/voice reference. Leave blank for the default SoulX singer.
VOICE_REFERENCE = ""
# Example: VOICE_REFERENCE = "/content/drive/MyDrive/Musicm8/voice_reference.wav"

VOCALS = True
VOCAL_STEPS = 24
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Musicm8")
WORK = ROOT / "work"
AUDIO = ROOT / "audio"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
WORK.mkdir(parents=True, exist_ok=True); AUDIO.mkdir(parents=True, exist_ok=True)

if (REPO / ".git").exists():
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","1","origin","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"reset","--hard","origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-ai.txt"], check=True)
subprocess.run(["apt-get","update","-qq"], check=True)
subprocess.run(["apt-get","install","-y","-qq","ffmpeg","fluidsynth","fluid-soundfont-gm"], check=False)
# Prefer MuseScore General when the Colab Ubuntu image provides the package; FluidR3 remains a safe fallback.
subprocess.run(["apt-get","install","-y","-qq","musescore-general-soundfont"], check=False)

import torch
if not torch.cuda.is_available(): raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

lyrics_input = WORK / "user_lyrics_input.txt"
if CUSTOM_LYRICS.strip():
    lyrics_input.write_text(CUSTOM_LYRICS.strip()+"\n", encoding="utf-8")
    print("✍️ USER LYRICS MODE — exact words preserved")
else:
    if lyrics_input.exists(): lyrics_input.unlink()
    print("✍️ AI LYRICS MODE")

cmd = [
    sys.executable,"-u","ai_producer_workflow.py",
    "--root",str(ROOT),"--repo",str(REPO),
    "--idea",IDEA,"--bars",str(BARS),"--seed",str(SEED),
    "--ai-model",AI_MODEL,"--composer-candidates",str(COMPOSER_CANDIDATES),
    "--vocal-steps",str(VOCAL_STEPS)
]
if CUSTOM_LYRICS.strip(): cmd += ["--lyrics-file",str(lyrics_input)]
if VOICE_REFERENCE.strip(): cmd += ["--voice-reference",VOICE_REFERENCE.strip()]
if not VOCALS: cmd.append("--no-vocals")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
(PROJECT/"song_seed.txt").write_text(str(SEED), encoding="utf-8")

print("\n================ 📝 LYRICS REQUESTED ================\n")
lf=PROJECT/"lyrics.txt"
print(lf.read_text(encoding="utf-8") if lf.exists() else "No lyrics file")
print("======================================================\n")

cr=PROJECT/"composition_report.json"
if cr.exists():
    c=json.loads(cr.read_text())
    print("🧠 LEARNED COMPOSER")
    print("model:",c.get("composer"))
    print("selected candidate:",c.get("selected_candidate"))
    print("role notes:",c.get("normalization",{}).get("roles"))
    print("rule composer fallback:",c.get("rule_composer_fallback"))

status_path=PROJECT/"final_status.json"
status=json.loads(status_path.read_text()) if status_path.exists() else {}
print("\n🎯 FINAL STATUS",status.get("final_kind"))

from IPython.display import Audio, display
for label,p in [
    ("AI COMPOSED INSTRUMENTAL",PROJECT/"master_instrumental.wav"),
    ("SOULX SCORE GUIDE",PROJECT/"vocals/soulx_score_guide.wav"),
    ("QA-PASSED LEAD",PROJECT/"vocals/neural_lead_synced.wav")
]:
    if p.exists():
        print("\n🎵",label); display(Audio(str(p)))
final=PROJECT/"master.wav"
if status.get("final_kind")=="song_with_vocals" and final.exists():
    print("\n✅ FINAL SONG — VOCALS PASSED"); display(Audio(str(final)))
elif VOCALS:
    print("\n❌ VOCAL FINAL FAILED — showing instrumental safety copy, not pretending it contains words")
    if final.exists(): display(Audio(str(final)))
    log=PROJECT/"vocals/vocal_backend.log"
    if log.exists(): print("\n--- VOCAL BACKEND LOG TAIL ---\n"+"\n".join(log.read_text(errors="ignore").splitlines()[-100:]))
else:
    print("\n✅ FINAL INSTRUMENTAL")
    if final.exists(): display(Audio(str(final)))
print(f"\n🌱 Seed: {SEED}")

## 🎤 Vocal-only retry

Use this only after the AI-composed instrumental is worth keeping. It keeps the music untouched, rebuilds the singer against the **actual MIDI-LLM chord/lead tracks**, generates SoulX phrase chunks, runs pitch/word QA and remixes the vocal.

In [ ]:
# MUSICM8 V10 — VOCAL ONLY
import os, sys, json, subprocess
from pathlib import Path
from IPython.display import Audio, display
ROOT=Path('/content/drive/MyDrive/Musicm8'); REPO=Path('/content/Musicm8'); PROJECT=ROOT/'work/ai_projects/latest'
VOICE_REFERENCE=""
VOCAL_STEPS=24
seed_file=PROJECT/'song_seed.txt'; SEED=int(seed_file.read_text().strip()) if seed_file.exists() else 42
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
os.chdir(REPO)
cmd=[sys.executable,'-u','retry_vocals_v10.py','--root',str(ROOT),'--repo',str(REPO),'--steps',str(VOCAL_STEPS),'--seed',str(SEED)]
if VOICE_REFERENCE.strip(): cmd += ['--voice-reference',VOICE_REFERENCE.strip()]
subprocess.run(cmd,check=True)
print('\n================ 📝 LYRICS USED ================\n'+(PROJECT/'lyrics.txt').read_text()+'=================================================\n')
for label,p in [('SOULX GUIDE',PROJECT/'vocals/soulx_score_guide.wav'),('QA-PASSED LEAD',PROJECT/'vocals/neural_lead_synced.wav'),('FINAL SONG',PROJECT/'master.wav')]:
    if p.exists(): print('\n🎵',label); display(Audio(str(p)))

## Diagnostics

If either the composer or singer fails, this prints the useful reports and backend tail.

In [ ]:
from pathlib import Path
project=Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
for p in [project/'composition_report.json',project/'soundfont_render_report.json',project/'final_status.json',project/'vocals/vocal_status.json',project/'vocals/vocal_quality.json',project/'vocals/vocal_word_quality.json']:
    if p.exists(): print('\n---',p.name,'---\n'+p.read_text())
log=project/'vocals/vocal_backend.log'
if log.exists(): print('\n--- SOULX BACKEND LOG TAIL ---\n'+'\n'.join(log.read_text(errors='ignore').splitlines()[-180:]))